# A full business solution

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-6-luna'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://bukovel.com/")
links

['https://eco.bukovel.com/',
 'https://bukovel.com/b24/hotels',
 '/discounts',
 '/entertainments',
 'https://invest.bukovel.com',
 'https://cooperation.bukovel.com/',
 'https://bukovel.com/b24/account/profile',
 'https://bukovel.com/b24/basket',
 'https://bukovel.com/b24/hotels',
 'https://bukovel.com/b24/skipass',
 'https://bukovel.com/b24/club-cards',
 'https://bukovel.com/aquapark',
 'https://bukovel.com/b24/tourist-center',
 'https://medicalbukovel.com.ua/',
 'https://bukovel.com/b24/insurance',
 '/cams',
 'https://bukovel.com/uk/information/maps',
 'https://bukovel.com/news/2-peredprodazh-ski-pasiv-vidkrito',
 'https://bukovel.com/b24/tourist-center',
 'https://bukovel.com/news/osin-u-bukovel-shche-vigidnisha-z-mastercard',
 'https://bukovel.com/news/rannie-broniuvannia-zimi-202627-u-bukovel-hotel-vidkrito',
 'https://bukovel.com/aquapark#prices',
 'https://bukovel.com/sustainability',
 'https://bukovel.com/summer/2-rodelban-speed-fun',
 'https://bukovel.com/summer/etnopark-gutsul

## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://bukovel.com/"))


Here is the list of links on the website https://bukovel.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://eco.bukovel.com/
https://bukovel.com/b24/hotels
/discounts
/entertainments
https://invest.bukovel.com
https://cooperation.bukovel.com/
https://bukovel.com/b24/account/profile
https://bukovel.com/b24/basket
https://bukovel.com/b24/hotels
https://bukovel.com/b24/skipass
https://bukovel.com/b24/club-cards
https://bukovel.com/aquapark
https://bukovel.com/b24/tourist-center
https://medicalbukovel.com.ua/
https://bukovel.com/b24/insurance
/cams
https://bukovel.com/uk/information/maps
https://bukovel.com/news/2-peredprodazh-ski-pasiv-vidkrito
https://bukovel.com/b24/tourist-center
https://bukovel.com/news/osin-u-bukovel-shche-vigidnisha-z-mastercard
https://bukovel.com/news/rannie-broniuvannia-z

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links


In [8]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [9]:
select_relevant_links("https://bukovel.com/")

Selecting relevant links for https://bukovel.com/ by calling gpt-6-luna
Found 13 relevant links


{'links': [{'type': 'about page', 'url': 'https://bukovel.com/sustainability'},
  {'type': 'careers page', 'url': 'https://job.bukovel.com/'},
  {'type': 'investment', 'url': 'https://invest.bukovel.com'},
  {'type': 'cooperation', 'url': 'https://cooperation.bukovel.com/'},
  {'type': 'hotels', 'url': 'https://bukovel.com/b24/hotels'},
  {'type': 'entertainment', 'url': 'https://bukovel.com/entertainments'},
  {'type': 'aquapark', 'url': 'https://bukovel.com/aquapark'},
  {'type': 'summer activities', 'url': 'https://bukovel.com/summer'},
  {'type': 'medical services', 'url': 'https://medicalbukovel.com.ua/'},
  {'type': 'contacts', 'url': 'https://bukovel.com/contacts'},
  {'type': 'Facebook', 'url': 'https://facebook.com/bukovel'},
  {'type': 'Instagram', 'url': 'https://www.instagram.com/Bukovel/'},
  {'type': 'mobile app',
   'url': 'https://apps.apple.com/app/bukovel-24/id1443785475'}]}

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'company brand', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'X', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://bukovel.com/"))

Selecting relevant links for https://bukovel.com/ by calling gpt-6-luna
Found 13 relevant links
## Landing Page:

Буковель Bukovel. Всесезонний курорт у серці Карпат.

ЕКОЛОГІЯ BUKOVEL
AQI
31
Норма
|
pH 7.2
|
Аквапарк
Норма
Готелі
Акції
Дозвілля Bukovel
Партнерам
INVEST BUKOVEL
ПАРТНЕРСТВО З БУКОВЕЛЬ
UA
EN
PL
RO
Готелі
Скі-паси
Акція
Сезонні картки
Акція
Аквапарк "Mavka"
Туристичний центр
Медичний центр
Страхування
Камери
Карти курорту
Всесезонний курорт Буковель - Відпочинок у серці Карпат - Bukovel
Осінь у Bukovel
ще вигідніша
з Mastercard
Детальніше
Туристичний Центр:
зручне бронювання онлайн
Детальніше
Раннє бронювання
зими відкрито!
Детальніше
Передпродаж скі-пасів відкрито!
Детальніше
Аквапарк MAVKA
Водна стихія
Буковеля
Купити квитки
Сталий розвиток Bukovel
Детальніше
Активний відпочинок
РОДЕЛЬБАН
ЕТНОПАРК "ГУЦУЛ ЛЕНД"
АКВАПАРК "MAVKA"
КОЛЕСО ОГЛЯДУ
ТУРИСТИЧНИЙ ЦЕНТР
Всі літні розваги
Новини
Loading...
Loading...
Loading...
Loading...
Loading...
Loading...
Loading...
Loading...


In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks. Respond in Ukrainian language.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks. Answer in Ukrainian\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("Bukovel", "https://bukovel.com/")

Selecting relevant links for https://bukovel.com/ by calling gpt-6-luna
Found 11 relevant links


'\nYou are looking at a company called: Bukovel\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks. Answer in Ukrainian\n\n\n## Landing Page:\n\nБуковель Bukovel. Всесезонний курорт у серці Карпат.\n\nЕКОЛОГІЯ BUKOVEL\nAQI\n31\nНорма\n|\npH 7.2\n|\nАквапарк\nНорма\nГотелі\nАкції\nДозвілля Bukovel\nПартнерам\nINVEST BUKOVEL\nПАРТНЕРСТВО З БУКОВЕЛЬ\nUA\nEN\nPL\nRO\nГотелі\nСкі-паси\nАкція\nСезонні картки\nАкція\nАквапарк "Mavka"\nТуристичний центр\nМедичний центр\nСтрахування\nКамери\nКарти курорту\nВсесезонний курорт Буковель - Відпочинок у серці Карпат - Bukovel\nТуристичний Центр:\nзручне бронювання онлайн\nДетальніше\nОсінь у Bukovel\nще вигідніша\nз Mastercard\nДетальніше\nРаннє бронювання\nзими відкрито!\nДетальніше\nПередпродаж скі-пасів відкрито!\nДетальніше\nАквапарк MAVKA\nВодна стихія\nБуковеля\nКупити квитки\nСталий розвиток Bukovel\nДетальніше\nАктивний відпочинок

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("Bukovel", "https://bukovel.com/")

Selecting relevant links for https://bukovel.com/ by calling gpt-6-luna
Found 13 relevant links


# Bukovel — всесезонний курорт у серці Карпат

Bukovel поєднує відпочинок, активні розваги, гостинність і турботу про довкілля. Курорт розташований у Карпатах і пропонує враження для різних сезонів та інтересів.

## Відпочинок і розваги

Серед можливостей Bukovel — готелі, гірськолижний відпочинок, аквапарк **MAVKA**, медичний центр, туристичний центр, а також літні розваги: родельбан, етнопарк **«Гуцул Ленд»** і колесо огляду. Для гостей доступні бронювання онлайн, скі-паси, сезонні картки, страхування та карти курорту.

## Сталий розвиток

Турбота про природу Карпат — частина щоденної роботи курорту. Bukovel розвиває сортування відходів, збільшує використання сонячної енергії та знайомить гостей із природою й культурною спадщиною регіону. На сайті також публікуються показники екологічного моніторингу, зокрема якості повітря та води.

## Люди й культура

У блозі Bukovel розповідає про працівників курорту, локальні громади, гуцульську спадщину, природні локації Карпат і нові ініціативи. Це погляд на Bukovel не лише як на місце відпочинку, а і як на частину регіону з власною історією та характером.

## Кар’єра в Bukovel

Курорт пропонує вакансії в різних напрямах: готельному бізнесі, ресторанах, прокаті спорядження, інформаційному центрі, касах, медичному центрі, аквапарку та сфері розваг. Є варіанти роботи в Поляниці, Яремче, Івано-Франківську та Києві. Якщо відповідної вакансії немає, кандидати можуть надіслати інформацію про себе.

## Для партнерів та інвесторів

Bukovel запрошує до партнерства й окремо представляє напрям **INVEST BUKOVEL**. Курорт поєднує туристичну інфраструктуру, сервіс і розвиток дестинації в Карпатах.

In [21]:
create_brochure("HuggingFace", "https://huggingface.co" )

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 9 relevant links


# Hugging Face

### Платформа, де спільнота машинного навчання створює майбутнє

Hugging Face об’єднує спільноту ШІ для спільної роботи над моделями, наборами даних і застосунками. Це платформа, де можна знаходити, створювати й розвивати рішення машинного навчання.

## Що пропонує платформа

- **Моделі:** доступ до каталогу з понад 2 мільйонами моделей.
- **Застосунки Spaces:** понад мільйон демонстрацій і ШІ-застосунків, які можна досліджувати.
- **Набори даних:** каталог із понад 500 тисячами наборів даних.
- **Інструменти для команд і бізнесу:** корпоративні плани, підтримка, Inference Providers та Inference Endpoints, а також Storage Buckets.
- **Спільнота й навчання:** документація, форум, Discord, GitHub, блог і матеріали для навчання.

## Для кого

**Розробникам і дослідникам** — щоб знаходити моделі й дані, створювати застосунки та співпрацювати з ML-спільнотою.

**Компаніям і командам** — щоб використовувати інфраструктуру та сервіси Hugging Face, а також отримувати корпоративну підтримку.

**Інвесторам і партнерам** — Hugging Face розвиває платформу для відкритої співпраці в галузі машинного навчання та об’єднує широку екосистему моделей, даних і застосунків.

## Культура та кар’єра

У центрі діяльності Hugging Face — спільнота, співпраця й доступ до інструментів машинного навчання. Інформації про внутрішню культуру компанії, відкриті вакансії чи умови роботи надані сторінки не містять.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("Bukovel", "https://bukovel.com/")

Selecting relevant links for https://bukovel.com/ by calling gpt-6-luna
Found 8 relevant links


# Буковель — всесезонний курорт у серці Карпат

Короткий рекламно‑інформаційний буклет для відпочивальників, інвесторів і кандидатів на роботу.

## Про курорт
Буковель — найбільший український гірськолижний та велнес‑курорт, який працює цілий рік. Розташований у Карпатах (с. Поляниця) і поєднує лижні траси, оздоровчі послуги та сімейні розваги з турботою про природу і локальну культуру.

## Послуги та розваги
- Готелі та розміщення (бронювання онлайн через Туристичний центр).
- Скі‑паси та сезонні картки (доступні передпродажі).
- Аквапарк MAVKA — водні атракціони та сімейний відпочинок.
- Рodelьбан (роллерні/санні траси), колесо огляду, етнопарк «Гуцул Ленд».
- Медичний центр, страхування, камери зберігання, детальні карти курорту.
- Літні і зимові активності, тематичні події та культурні програми.

## Стійкий розвиток та екологія
- Офіс сталого розвитку курорту: sustainability — частина щоденної роботи.
- Екологічні метрики на сайті: AQI ≈ 40 (норма), показник pH води ≈ 7.2.
- Розвиток сонячної енергетики — нові сонячні панелі потужністю 104 кВт.
- Програми сортування відходів, ініціативи збереження біорізноманіття (приклад — про види місцевих бджіл).
- Соціальні та екокультурні проекти: підтримка локальної спадщини Гуцульщини, артоб’єкти, «Літературна вітальня», Lost & Found бокси, знижки за багаторазовий посуд у кав’ярнях.

## Клієнти та партнери
- Цільова аудиторія: сім’ї, лижники і сноубордисти, шанувальники активного літнього відпочинку, іноземні та вітчизняні туристи.
- Партнерські програми та інвестпроекти (Invest Bukovel) для розвитку інфраструктури і розширення сервісів.
- Туристичний центр для зручного онлайн‑бронювання та інформації.

## Кар’єра та робота
- Велика кількість вакансій у різних підрозділах: прокат спорядження, інформаційний центр, готелі («Буковель», «Едельвейс» у Яремчі), РЦ «Бука», каси ТК, ресторани (наприклад NUMO), Bukovel Medical Center, аквапарк MAVKA, родельбан та ін.
- Локації праці: ТК "Буковель" (Поляниця), м. Івано‑Франківськ, м. Київ (центральний офіс), м. Яремче.
- Приклад вакансії: двірник в аквапарк MAVKA — заробітна плата 18 000–30 000 грн.
- Не знайшли підходящої вакансії? Курорт приймає резюме «про себе» — можливість потрапити до бази кандидатів.

## Культура компанії
- Акцент на сталому розвитку, відповідальному ставленні до природи та локальній культурі.
- Підтримка творчих і освітніх ініціатив (арт‑об’єкти, культурні події, блог курорту з інтерв’ю та репортажами).
- Команда, яка цінує інновації (офіс sustainability, цифрові сервіси), гостинність і професіоналізм у сервісі.

## Останні ініціативи (приклади)
- Встановлення додаткових сонячних панелей (104 кВт).
- Запуск програми сортування відходів та переробки паперу.
- Відкриття боксів Lost & Found.
- Культурні та просвітницькі події: відкриття артоб’єктів, літературна вітальня.

## Як долучитися, забронювати, інвестувати
- Купуйте квитки та скі‑паси, бронюйте готелі онлайн через Туристичний центр.
- Слідкуйте за акціями (раннє бронювання, спеціальні пропозиції).
- Для співпраці або інвестицій — розділ «INVEST BUKOVEL» та партнерські контакти на офіційному сайті.
- Для роботи — перегляньте вакансії на сторінці «Робота в Буковелі» або надішліть своє резюме, якщо підходящої позиції немає.

Підписуйтеся на розсилку курорту, щоб дізнаватися про новини, акції та відкриття першими. Приїжджайте у Буковель — відчуйте Карпати відповідально та з комфортом.

In [28]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-6-luna
Found 9 relevant links


# Hugging Face — Спільнота штучного інтелекту, яка створює майбутнє

Коротко
- Платформа для спільної розробки ML: моделі, датасети, додатки та інфраструктура.
- Велика відкрита екосистема: понад 2M моделей, понад 1M додатків (Spaces) та 500k+ датасетів.
- Місце зустрічі дослідників, інженерів і підприємств — від експериментів до продуктивних рішень.

Про компанію
- Місія: робити машинне навчання відкритим, доступним і зручним для співпраці.
- Центр активностей: модельні репозиторії, датасети, інтерактивні додатки (Spaces), сховища (Buckets), документація, блог і щоденні наукові огляди.
- Активність спільноти видно в постійних оновленнях моделей, датасетів і просторів (наприклад: LiquidAI/LFM3-1.7B, mistralai/Mistral-Large-4-1T-A52B, nvidia/NVIDIA-Nemotron-3.5).

Продукти та сервіси
- Models: каталог із мільйонами моделей для NLP, CV, аудіо та мультимодальних задач.
- Datasets: великий набір публічних датасетів для навчання й оцінки.
- Spaces: хостинг інтерактивних додатків (генерація зображень, відео, агентні системи, демо).
- Buckets: сховище для даних і артефактів з інтеграцією в робочі процеси.
- Inference Endpoints & Providers: продуктивний inference для виробничих систем.
- Hugging Face PRO, Enterprise Support: інструменти та підтримка для команд і корпоративних клієнтів.

Спільнота та ресурси
- Активні канали: Discord, форум, GitHub і блоги з технічними статтями та оглядами.
- Освітні матеріали: документація, навчальні курси, щоденні наукові статті (Daily Papers).
- Відкритий розвиток: сотні тисяч оновлень і внесків від спільноти — як у відкритих проектах, так і в приватних організаціях.

Культура
- Відкритість і співпраця: орієнтація на open source, відкриті репозиторії та обмін знаннями.
- Науково-інженерний підхід: блог і публікації про нові методи, інструменти та практики (наприклад, токенізація, агентні системи, post-training).
- Орієнтація на результат: від прототипів у Spaces до промислових deployment через Inference Endpoints і Enterprise-рішення.
- Спільнота над продуктом: користувачі та розробники активно взаємодіють, тестують і вносять покращення.

Клієнти та сфери застосування
- Хто користується: дослідницькі лабораторії, стартапи, продуктові команди, великі компанії, які потребують кастомної інференс-інфраструктури та підтримки.
- Сфери: генерація тексту й зображень, семантичний пошук, аудіо/транскрипція, агентні системи, промислові ML-пайплайни, симуляції і 3D-активи.
- Рішення для бізнесу: Team & Enterprise плани, PRO-акаунти, інтерфейси для продуктивного inference і зберігання даних.

Кар’єра та можливості приєднатись
- Типові ролі: ML-інженери, дослідники, інженери інфраструктури, продукт-менеджери, інженери підтримки enterprise.
- Робота в відкритій та динамічній команді з фокусом на інноваціях і публікаціях.
- Як почати: відвідайте huggingface.co, підключіться до спільноти в Discord та на GitHub, слідкуйте за оновленнями в блозі і розділах Team & Enterprise для оголошень про вакансії.

Як почати / Контакт
- Досліджуйте: browse 2M+ models, 1M+ apps, 500k+ datasets на huggingface.co.
- Спробуйте Spaces для швидких демо і прототипів.
- Для бізнесу: ознайомтесь з Team & Enterprise, Hugging Face PRO, Inference Endpoints і Storage Buckets.
- Приєднатись до спільноти: Discord, Forum, GitHub та блог компанії.

Hugging Face — це екосистема, де відкритість, наука і продуктові рішення збираються разом, щоб пришвидшувати інновації в галузі ШІ.